This script generates a pan-India raster of 'distance to nearest settlement'. The settlements mask is obtained by taking a union of WRIS's settlement polygons (https://indiawris.gov.in/wris/#/geoSpatialData: navigate to 'Settlement Extent and Location') with IndiaSAT LULC's Built-up class, moded over the past 3 years. The rasters are generated one state at a time.

# Set up

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
import ee
import geemap
import pandas as pd

ee.Authenticate()
ee.Initialize(project='ee-mtpictd')

# Config

In [ ]:
# State names and base asset path
states = [
    'andhra_pradesh', 'arunachal_pradesh', 'assam', 'bihar', 'chhattisgarh',
    'gujarat', 'haryana', 'himachal_pradesh', 'jharkhand',
    'delhi', 'chandigarh', 'goa',
    # 'jammu_and_kashmir',
    'karnataka', 'kerala', 'madhya_pradesh', 'maharashtra', 'manipur',
    'meghalaya', 'mizoram', 'nagaland', 'orissa', 'punjab', 'rajasthan',
    'sikkim', 'tamil_nadu', 'telangana', 'tripura', 'uttar_pradesh',
    'uttarakhand', 'west_bengal'
]

# J&K is missing; GAUL doesn't have a state boundary for it
gaulNames = {
    'andhra_pradesh': 'Andhra Pradesh',
    'arunachal_pradesh': 'Arunachal Pradesh',
    'assam': 'Assam',
    'bihar': 'Bihar',
    'chhattisgarh': 'Chhattisgarh',
    'gujarat': 'Gujarat',
    'haryana': 'Haryana',
    'himachal_pradesh': 'Himachal Pradesh',
    'jharkhand': 'Jharkhand',
    'delhi': 'Delhi',
    'chandigarh': 'Chandigarh',
    'goa': 'Goa',
    'karnataka': 'Karnataka',
    'kerala': 'Kerala',
    'madhya_pradesh': 'Madhya Pradesh',
    'maharashtra': 'Maharashtra',
    'manipur': 'Manipur',
    'meghalaya': 'Meghalaya',
    'mizoram': 'Mizoram',
    'nagaland': 'Nagaland',
    'orissa': 'Orissa',
    'punjab': 'Punjab',
    'rajasthan': 'Rajasthan',
    'sikkim': 'Sikkim',
    'tamil_nadu': 'Tamil Nadu',
    'telangana': 'Andhra Pradesh', # GAUL 2015 is not updated to reflect Telangana
    'tripura': 'Tripura',
    'uttar_pradesh': 'Uttar Pradesh',
    'uttarakhand': 'Uttarakhand',
    'west_bengal': 'West Bengal'
}

In [ ]:
lulcBasePath = 'projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_'
wrisBasePath = 'projects/ee-mtpictd/assets/anoushka/wris_settlements/'

indiaACZs = ee.FeatureCollection("projects/ee-mtpictd/assets/harsh/Agroclimatic_regions")
india = indiaACZs

roi = india
scale = 100

# Utils

In [ ]:
def getStateRoi(state):
    """Return GAUL level-1 Feature for a given state."""
    gaul_name = gaulNames[state]
    return (
        ee.FeatureCollection('FAO/GAUL/2015/level1')
        .filter(ee.Filter.eq('ADM1_NAME', gaul_name))
        .first()
        .set('state', state)
    )

def getImgCollection(folder_path):
  asset_list = ee.data.listAssets({'parent': folder_path})['assets']
  image_list = [ee.Image(asset['name']) for asset in asset_list]
  return ee.ImageCollection(image_list)

def emptyFolder(folder_path):
  assets = ee.data.listAssets({'parent': folder_path})
  for asset in assets['assets']:
      asset_id = asset['name']
      print(f'Deleting: {asset_id}')
      ee.data.deleteAsset(asset_id)
  print('All assets deleted.')

def deleteAsset(asset_id):
  ee.data.deleteAsset(asset_id)

def listAssets(folder_path):
  assets = ee.data.listAssets({'parent': folder_path})
  for asset in assets['assets']:
      print(asset['name'])
  return assets['assets']

# Exporting settlement rasters

## Get wris mask

In [ ]:
# Load and merge WRIS settlements for all states
wris_fc_list = [ee.FeatureCollection(wrisBasePath + s) for s in states]
wrisSettlements = ee.FeatureCollection(wris_fc_list).flatten()

In [ ]:
# Rasterize to sparse masked image
wrisMask = (
    ee.Image()
    .byte()
    .paint(wrisSettlements, 1)
    .clip(india.geometry())
)

# Convert to explicit 0–1 image
wris01 = wrisMask.unmask(0)
wris01.getInfo()

Output hidden; open in https://colab.research.google.com to view.

## Get indiasat masks

In [ ]:
# For IndiaSAT LULC based mask, use modal output of past 3 years
year = 2023
years = [year-2, year-1, year]

lulc_images = [
    ee.Image(f"{lulcBasePath}{y}_{y+1}").toByte()
    for y in years
]

lulcMode = (
    ee.ImageCollection(lulc_images)
    .toBands()
    .reduce(ee.Reducer.mode())
)

# Mask to class == 1, then convert to 0–1
lulcMask = lulcMode.updateMask(lulcMode.eq(1))
lulc01 = lulcMask.unmask(0).clip(india.geometry())


## Union and get distance image

In [ ]:
# Take union of WRIS and moded IndiaSAT
unionMask = wris01.max(lulc01)

print(unionMask.getInfo())

{'type': 'Image', 'bands': [{'id': 'constant', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [325378, 337548], 'origin': [803, 0], 'crs': 'EPSG:4326', 'crs_transform': [8.983152841195215e-05, 0, 68.11403776613543, 0, -8.983152841195215e-05, 37.0783226781469]}]}


In [ ]:
# Get distance-from-settlements image pan-India
distImgIndia = (
    unionMask
    .fastDistanceTransform()
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .clip(india.geometry())
)

print(distImgIndia.getInfo())

Output hidden; open in https://colab.research.google.com to view.

## Sanity checking

In [ ]:
Map = geemap.Map()
Map.addLayer(unionMask, {'min': 0, 'max': 1, 'palette': ['black', 'yellow']}, 'Union seeds')
Map.addLayer(distImgIndia, {'min': 0, 'max': 10000, 'palette': ['white', 'teal', 'green']}, 'Distance')
Map.centerObject(india, 4)
Map

Map(center=[23.22103836473429, 79.4609888487962], controls=(WidgetControl(options=['position', 'transparent_bg…

The provided color (teal) is invalid. Using the default black color.
'#teal' is not in web format. Need 3 or 6 hex digit.


## Export state wise

I used GAUL 2015's state boundaries to export the distance-from-nearest-settlement raster state by state. But these boundaries are not completely accurate (e.g. J&K is not available). So I had to do additional gap-filling later, which can be avoided if you use a better map of India's states. Just edit the list `states` in Config and the function `getStateRoi` in Utils.


In [ ]:
for state in states:
  stateRoi = getStateRoi(state)
  geom = stateRoi.geometry()
  task = ee.batch.Export.image.toAsset(
        image=distImgIndia.clip(geom),
        description=f'dist_to_settlements_{state}',
        assetId=f'projects/ee-mtpictd/assets/anoushka/settlement_distances/{state}',
        region=geom,
        scale=scale,
        crs='EPSG:4326',
        maxPixels=1e13
  )
  task.start()
  print(f'Export task started for {state}')

Export task started for andhra_pradesh
Export task started for arunachal_pradesh
Export task started for assam
Export task started for bihar
Export task started for chhattisgarh
Export task started for gujarat
Export task started for haryana
Export task started for himachal_pradesh
Export task started for jharkhand
Export task started for delhi
Export task started for chandigarh
Export task started for goa
Export task started for karnataka
Export task started for kerala
Export task started for madhya_pradesh
Export task started for maharashtra
Export task started for manipur
Export task started for meghalaya
Export task started for mizoram
Export task started for nagaland
Export task started for orissa
Export task started for punjab
Export task started for rajasthan
Export task started for sikkim
Export task started for tamil_nadu
Export task started for telangana
Export task started for tripura
Export task started for uttar_pradesh
Export task started for uttarakhand
Export task start

## Gap filling

In [ ]:
stateRoiList = []
for state in states:
  if state == 'jammu_and_kashmir' or state == 'ladakh':
    continue
  stateRoiList.append(getStateRoi(state))

stateRois = ee.Feature(ee.FeatureCollection(stateRoiList).union().first())
indiaRoi = ee.Feature(indiaACZs.union().first())
diff = indiaRoi.difference(stateRois)

print(diff.getInfo())

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
Map = geemap.Map()
simple = diff.geometry().simplify(1000)
Map.addLayer(simple, {}, 'Difference')
Map.addLayer(india, {}, 'India Boundary')
Map.centerObject(india, 5)

Map

Map(center=[23.22103836473429, 79.4609888487962], controls=(WidgetControl(options=['position', 'transparent_bg…

In [ ]:
img = distImgIndia.clip(diff.geometry())
task = ee.batch.Export.image.toAsset(
    image=img,
    description=f'Settlement_Distance_diff',
    assetId=f'projects/ee-mtpictd/assets/anoushka/settlement_distances/diff',
    region=diff.geometry(),
    scale=scale,
    crs='EPSG:4326',
    maxPixels=1e13
)
task.start()
